In [ ]:
from pathlib import Path
DATA_DIR = Path("../data").resolve() / "melb_jun_25"                 
LISTINGS_CSV = DATA_DIR / "listings.csv"
REVIEWS_CSV  = DATA_DIR / "reviews.csv"
OUTPUT_DIR = Path("../output").resolve()

In [ ]:
reviews_df = pd.read_csv(REVIEWS_CSV)

In [ ]:
def text_stats(s):
    words = s.split()
    wc = len(words)
    char_count = len(s)
    avg_word_len = (sum(len(w) for w in words) / wc) if wc>0 else 0.0
    exclam = s.count("!")
    question = s.count("?")
    uppercase_words = sum(1 for w in words if any(c.isupper() for c in w))
    uppercase_ratio = uppercase_words / wc if wc>0 else 0.0
    stopwords = sum(1 for w in words if w.lower() in ENGLISH_STOP_WORDS)
    stopword_ratio = stopwords / wc if wc>0 else 0.0
    return {
        "word_count": wc,
        "char_count": char_count,
        "avg_word_len": avg_word_len,
        "exclam_count": exclam,
        "question_count": question,
        "uppercase_ratio": uppercase_ratio,
        "stopword_ratio": stopword_ratio
    }

In [ ]:
def preprocess_reviews_text_feat(reviews_df):
    reviews_df["comments_clean"] = reviews_df.apply(lambda x: clean_text(x["comments"]),axis=1)
    reviews_df = reviews_df.join(reviews_df.apply(lambda x: text_stats(x['comments_clean']), axis=1, result_type="expand"))
    return reviews_df

In [ ]:
def preprocess_reviews_tabular_feats(reviews_df):
    reviews_df["review_date"] = pd.to_datetime(reviews_df["date"], errors="coerce")
    max_date = reviews_df["review_date"].max()
    reviews_df["review_age_days"] = (max_date - reviews_df["review_date"]).dt.days
    reviews_df["review_month"] = reviews_df["review_date"].dt.month
    reviews_df["review_weekday"] = reviews_df["review_date"].dt.weekday
    reviews_df["review_year"] = reviews_df["review_date"].dt.year
    return reviews_df

In [ ]:
REVIEW_FEATS = [
                'word_count', 'char_count', 'avg_word_len',
                'exclam_count', 'question_count', 'uppercase_ratio', 'stopword_ratio'
                ]
REVIEW_FEAT_AGGREGATIONS = ["mean","median","max","min"]

In [ ]:
reviews_df = (
                reviews_df
                .pipe(preprocess_reviews_text_feat)
                .pipe(preprocess_reviews_tabular_feats)
            )

In [ ]:
def derive_aggregate_review_text_metrics_by_listing(reviews_df, review_feats, metrics):
    aggregate_metrics = reviews_df.groupby('listing_id')[REVIEW_FEATS].agg(metrics)
    aggregate_metrics.columns = ['_'.join(col).strip() for col in aggregate_metrics.columns.values]
    aggregate_metrics = aggregate_metrics.reset_index()
    return aggregate_metrics

In [ ]:
review_metrics = derive_aggregate_review_text_metrics_by_listing(reviews_df, review_feats=REVIEW_FEATS,metrics=REVIEW_FEAT_AGGREGATIONS)